# 1. 加载训练数据


In [ ]:
import time
import tqdm
import gzip
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple

%matplotlib inline


In [ ]:
train_images_filename = "data/train-images-idx3-ubyte.gz"
train_labels_filename = "data/train-labels-idx1-ubyte.gz"
test_images_filename = "data/t10k-images-idx3-ubyte.gz"
test_labels_filename = "data/t10k-labels-idx1-ubyte.gz"

validation_size = 5000


def read_mnist_image_set(images_filename):
    with gzip.GzipFile(images_filename, "rb") as gz:
        magic = int.from_bytes(gz.read(4), "big")
        assert magic == 2051, "Not an MNIST image set"

        num_images = int.from_bytes(gz.read(4), "big")
        rows = int.from_bytes(gz.read(4), "big")
        cols = int.from_bytes(gz.read(4), "big")

        data = np.frombuffer(gz.read(num_images * rows * cols), dtype=np.uint8)
        data = np.reshape(data, (num_images, rows * cols))
        return data


def read_mnist_label_set(labels_filename):
    with gzip.GzipFile(labels_filename, "rb") as gz:
        magic = int.from_bytes(gz.read(4), "big")
        assert magic == 2049, "Not an MNIST label set"

        num_labels = int.from_bytes(gz.read(4), "big")
        labels = np.frombuffer(gz.read(num_labels), dtype=np.uint8)
        return labels


print("\nLoading train set ...")
train_data = read_mnist_image_set(train_images_filename) / np.float32(255)
train_labels = read_mnist_label_set(train_labels_filename)
train_data, val_data = train_data[:-validation_size], train_data[-validation_size:]
train_labels, val_labels = (
    train_labels[:-validation_size],
    train_labels[-validation_size:],
)
print(f"train_data:   [{str(train_data.dtype)}] {train_data.shape}")
print(f"train_labels: [{str(train_labels.dtype)}] {train_labels.shape}")
print(f"val_data:     [{str(val_data.dtype)}] {val_data.shape}")
print(f"val_labels:   [{str(val_labels.dtype)}] {val_labels.shape}")

print("\nLoading test set ...")
test_data = read_mnist_image_set(test_images_filename) / np.float32(255)
test_labels = read_mnist_label_set(test_labels_filename)
print(f"test_data:   [{str(test_data.dtype)}] {test_data.shape}")
print(f"test_labels: [{str(test_labels.dtype)}] {test_labels.shape}")


In [ ]:
# Preview dataset

_ = plt.figure(figsize=(6, 6))
_ = plt.title("MNIST Preview")
for label in range(10):
    for img_index, img_data in enumerate(train_data[train_labels == label][:10]):
        _ = plt.imshow(
            img_data.reshape(28, 28),
            "gray",
            vmin=0,
            vmax=1,
            interpolation="nearest",
            extent=(img_index, img_index + 1, label, label + 1),
        )
_ = plt.xticks([])
_ = plt.yticks(np.arange(10) + 0.5, range(10))
_ = plt.xlim(0, 10)
_ = plt.ylim(0, 10)

# 2. 训练


In [ ]:
# You may freely modify any given template code as you see fit.

batch_size = 100
max_epoch = 10
lr = 0.001  # learning rate
momentum = 0.9  # momentum

W = np.random.randn(784, 10) * 0.001  # weight
b = np.zeros(10)  # bias
v_W = np.zeros_like(W)
v_b = np.zeros_like(b)

train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(max_epoch):
    steps_per_epoch = np.ceil(len(train_data) / batch_size).astype(int)
    progress_bar = tqdm.tqdm(range(steps_per_epoch))
    for step_index in progress_bar:
        # fetch next batch of training data
        x = train_data[
            step_index * batch_size : (step_index + 1) * batch_size
        ]  # train_data as x (shape: [batch_size, 784])
        y_true = train_labels[
            step_index * batch_size : (step_index + 1) * batch_size
        ]  # train_labels as y (shape: [batch_size,])
        y_true = np.eye(10)[y_true]  # make one-hot (shape: [batch_size, 10])

        # TODO: forward with softmax
        y_pred = np.exp(x @ W + b) / np.sum(
            np.exp(x @ W + b), axis=1, keepdims=True
        )  # h_k(x) in the slides

        # TODO: calculate cross-entropy loss & accuracy
        loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=1))
        accuracy = np.mean(np.argmax(y_pred, axis=1) == np.argmax(y_true, axis=1))

        train_losses.append(loss)
        train_accuracies.append(accuracy)
        progress_bar.set_description(f"loss {loss:.3f} acc {accuracy:.3f}")

        # TODO: update weights, don't forget momentum
        error = y_pred - y_true
        grad_W = x.T @ error / x.shape[0]
        grad_b = np.sum(error, axis=0) / x.shape[0]
        v_W = v_W * momentum - lr * grad_W
        v_b = v_b * momentum - lr * grad_b
        W = W + v_W
        b = b + v_b

    # test
    x = test_data
    y_true = test_labels
    y_true = np.eye(10)[y_true]  # make one-hot

    # TODO: test set: forward with softmax
    y_pred = np.exp(x @ W + b) / np.sum(
        np.exp(x @ W + b), axis=1, keepdims=True
    )  # h_k(x) in the slides

    # TODO: test set: calculate cross-entropy loss & accuracy
    loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=1))
    accuracy = np.mean(np.argmax(y_pred, axis=1) == np.argmax(y_true, axis=1))

    test_losses.append(loss)
    test_accuracies.append(accuracy)
    print(f"  - test loss: {loss:.3f} acc: {accuracy:.3f}")

# plot loss & accuracy curve
fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 4), dpi=300)
_ = a0.set_title("Losses")
_ = a0.plot(train_losses, label="train")
_ = a0.plot(
    (np.arange(max_epoch) + 1) * steps_per_epoch, test_losses, marker=".", label="test"
)
_ = a0.legend()
_ = a1.set_title("Accuracies")
_ = a1.plot(train_accuracies, label="train")
_ = a1.plot(
    (np.arange(max_epoch) + 1) * steps_per_epoch,
    test_accuracies,
    marker=".",
    label="test",
)
_ = a1.legend()

# 3. 实验


## 1、Softmax 模型


In [ ]:
class SoftmaxTrainer:
    """
    Softmax分类器训练器，使用带动量的随机梯度下降进行优化

    Attributes:
        input_dim (int): 输入特征维度，默认为784（28x28图像展开）
        output_dim (int): 输出类别数量，默认为10（MNIST的0-9数字）
        W (np.ndarray): 权重矩阵，形状为(input_dim, output_dim)
        b (np.ndarray): 偏置向量，形状为(output_dim,)
        v_W (np.ndarray): 权重的动量项，形状与W相同
        v_b (np.ndarray): 偏置的动量项，形状与b相同
        timer (dict): 训练过程计时器
    """

    def __init__(
        self,
        input_dim: int = 784,
        output_dim: int = 10,
        random_seed: int = 2025,
        verbose: bool = True,
    ) -> None:
        """
        初始化Softmax训练器

        Args:
            input_dim (int): 输入特征维度. 默认为784.
            output_dim (int): 输出类别数量. 默认为10.
            random_seed (int): 随机数种子. 默认为2025.
            verbose (bool): 是否展示运算数据. 默认为True.
        """

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.random_seed = random_seed
        self.verbose = verbose
        self._reset_parameters()
        self._reset_timer()

    def _reset_parameters(self) -> None:
        """
        重置模型参数到初始状态
        """

        np.random.seed(self.random_seed)
        self.W = np.random.randn(self.input_dim, self.output_dim) * 0.001
        self.b = np.zeros(self.output_dim)
        self.v_W = np.zeros_like(self.W)
        self.v_b = np.zeros_like(self.b)

    def _reset_timer(self) -> None:
        """
        重置计时器，初始化所有计时记录
        """

        self.timer = {
            "total": 0.0,
            "forward": 0.0,
            "backward": 0.0,
            "update": 0.0,
            "evaluation": 0.0,
            "epochs": [],
        }

    def _softmax(self, x: np.ndarray) -> np.ndarray:
        """
        softmax实现

        Args:
            x (np.ndarray): 输入矩阵，形状为(batch_size, output_dim)

        Returns:
            np.ndarray: softmax概率输出，形状与x相同
        """

        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    def _forward(self, x: np.ndarray) -> np.ndarray:
        """
        前向传播计算

        Args:
            x (np.ndarray): 输入数据，形状为(batch_size, input_dim)

        Returns:
            np.ndarray: 预测概率分布，形状为(batch_size, output_dim)
        """

        logits = x @ self.W + self.b
        return self._softmax(logits)

    def _compute_loss_accuracy(
        self, y_pred: np.ndarray, y_true: np.ndarray
    ) -> Tuple[float, float]:
        """
        计算交叉熵损失和准确率

        Args:
            y_pred (np.ndarray): 预测概率，形状为(batch_size, output_dim)
            y_true (np.ndarray): 真实标签，形状为(batch_size,)

        Returns:
            Tuple[float, float]: (损失值, 准确率)
        """

        y_true_onehot = np.eye(self.output_dim)[y_true]
        loss = -np.mean(np.sum(y_true_onehot * np.log(y_pred + 1e-18), axis=1))
        accuracy = np.mean(np.argmax(y_pred, axis=1) == y_true)
        return loss, accuracy

    def train(
        self,
        train_data: np.ndarray,
        train_labels: np.ndarray,
        val_data: np.ndarray,
        val_labels: np.ndarray,
        batch_size: int = 100,
        max_epoch: int = 10,
        lr: float = 0.001,
        momentum: float = 0.0,
    ) -> Dict[str, List[float]]:
        """
        训练Softmax分类器

        Args:
            train_data (np.ndarray): 训练数据，形状为(n_train, input_dim)
            train_labels (np.ndarray): 训练标签，形状为(n_train,)
            val_data (np.ndarray): 测试数据，形状为(n_val, input_dim)
            val_labels (np.ndarray): 测试标签，形状为(n_val,)
            batch_size (int, optional): 批大小. 默认为100.
            max_epoch (int, optional): 最大训练轮数. 默认为10.
            lr (float, optional): 学习率. 默认为0.001.
            momentum (float, optional): 动量系数. 默认为0.0.

        Returns:
            Dict[str, List[float]]: 包含训练过程的损失和准确率记录
        """

        np.random.seed(self.random_seed)
        self._reset_parameters()
        self._reset_timer()
        total_start_time = time.time()

        train_losses = []
        val_losses = []
        train_accuracies = []
        val_accuracies = []

        n_train = len(train_data)
        steps_per_epoch = np.ceil(n_train / batch_size).astype(int)

        for epoch in range(max_epoch):
            np.random.seed(self.random_seed + epoch)
            epoch_start_time = time.time()
            epoch_times = {
                "forward": 0.0,
                "backward": 0.0,
                "update": 0.0,
                "evaluation": 0.0,
            }

            # 打乱训练数据
            indices = np.random.permutation(n_train)
            train_data = train_data[indices]
            train_labels = train_labels[indices]

            if self.verbose:
                progress_bar = tqdm.tqdm(range(steps_per_epoch))
            else:
                progress_bar = range(steps_per_epoch)

            for step_index in progress_bar:
                # 获取批次数据
                start_idx = step_index * batch_size
                end_idx = min((step_index + 1) * batch_size, n_train)
                x = train_data[start_idx:end_idx]
                y_true = train_labels[start_idx:end_idx]

                # 前向传播
                forward_start = time.time()
                y_pred = self._forward(x)
                epoch_times["forward"] += time.time() - forward_start

                # 计算损失和准确率
                loss, accuracy = self._compute_loss_accuracy(y_pred, y_true)
                train_losses.append(loss)
                train_accuracies.append(accuracy)
                if self.verbose:
                    progress_bar.set_description(
                        f"Epoch {epoch + 1}/{max_epoch} "
                        f"Loss: {loss:.3f} "
                        f"Acc: {accuracy:.3f}"
                    )

                # 反向传播计算梯度
                backward_start = time.time()
                y_true_onehot = np.eye(self.output_dim)[y_true]
                error = y_pred - y_true_onehot
                grad_W = x.T @ error / x.shape[0]
                grad_b = np.sum(error, axis=0) / x.shape[0]
                epoch_times["backward"] += time.time() - backward_start

                # 参数更新
                update_start = time.time()
                self.v_W = momentum * self.v_W - lr * grad_W
                self.v_b = momentum * self.v_b - lr * grad_b
                self.W += self.v_W
                self.b += self.v_b
                epoch_times["update"] += time.time() - update_start

            # 在每个epoch结束后评估测试集
            eval_start = time.time()
            val_pred = self._forward(val_data)
            val_loss, val_accuracy = self._compute_loss_accuracy(val_pred, val_labels)
            val_losses.append(val_loss)
            val_accuracies.append(val_accuracy)
            epoch_times["evaluation"] = time.time() - eval_start

            # 记录epoch时间
            epoch_time = time.time() - epoch_start_time
            self.timer["epochs"].append(
                {
                    "epoch": epoch + 1,
                    "total_time": epoch_time,
                    "component_times": epoch_times.copy(),
                    "val_loss": val_loss,
                    "val_accuracy": val_accuracy,
                }
            )

            if self.verbose:
                print(
                    f"   - "
                    f"Val Loss: {val_loss:.3f}, "
                    f"Val Acc: {val_accuracy:.3f}, "
                    f"Time: {epoch_time:.2f}s"
                )
            else:
                if epoch == max_epoch:
                    print(
                        "Final result: "
                        f"{max_epoch} epochs - "
                        f"Val Loss: {val_loss:.3f}, "
                        f"Val Acc: {val_accuracy:.3f}, "
                        f"Time: {epoch_time:.2f}s"
                    )

        # 记录总时间
        self.timer["total"] = time.time() - total_start_time

        # 打印详细计时信息
        if self.verbose:
            self.print_timing_info(max_epoch)

        return {
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accuracies": train_accuracies,
            "val_accuracies": val_accuracies,
            "steps_per_epoch": steps_per_epoch,
        }

    def evaluate(
        self, test_data: np.ndarray, test_labels: np.ndarray
    ) -> Tuple[float, float]:
        """
        评估模型在测试集上的性能

        Args:
            test_data (np.ndarray): 测试数据，形状为(n_test, input_dim)
            test_labels (np.ndarray): 测试标签，形状为(n_test,)

        Returns:
            Tuple[float, float]: (测试损失, 测试准确率)
        """

        start_time = time.time()
        test_pred = self._forward(test_data)
        test_loss, test_accuracy = self._compute_loss_accuracy(test_pred, test_labels)
        self.timer["evaluation"] = time.time() - start_time

        return test_loss, test_accuracy

    def print_timing_info(self, max_epoch: int) -> None:
        """
        打印训练过程的详细计时信息

        Args:
            max_epoch (int): 总训练轮数
        """

        print("\n=== 训练计时信息 ===")
        print(f"总训练时间: {self.timer['total']:.2f}秒")
        print(f"总训练轮数: {max_epoch}")

        if self.verbose and self.timer["epochs"]:
            avg_epoch_time = np.mean([e["total_time"] for e in self.timer["epochs"]])
            print(f"平均每轮时间: {avg_epoch_time:.2f}秒")

            # 计算各组件时间占比
            component_times = {
                "forward": np.mean(
                    [e["component_times"]["forward"] for e in self.timer["epochs"]]
                ),
                "backward": np.mean(
                    [e["component_times"]["backward"] for e in self.timer["epochs"]]
                ),
                "update": np.mean(
                    [e["component_times"]["update"] for e in self.timer["epochs"]]
                ),
                "evaluation": np.mean(
                    [e["component_times"]["evaluation"] for e in self.timer["epochs"]]
                ),
            }

            print("\n=== 时间分布 ===")
            for component, time_spent in component_times.items():
                percentage = (time_spent / sum(component_times.values())) * 100
                print(f"  {component}: {time_spent:.2f}秒 ({percentage:.1f}%)")

        # 打印每轮详细时间
        if self.verbose:
            print("\n=== 每轮详细时间 ===")
            for epoch_info in self.timer["epochs"]:
                print(
                    f"Epoch {epoch_info['epoch']}: "
                    f"{epoch_info['total_time']:.2f}秒, "
                    f"Val Loss: {epoch_info['val_loss']:.3f}, "
                    f"Val Acc: {epoch_info['val_accuracy']:.3f}"
                )

    def plot_curves(self, results: Dict[str, List[float]], max_epoch: int = 10) -> None:
        """
        绘制训练和测试的损失与准确率曲线

        Args:
            results (Dict[str, List[float]]): 训练结果字典
            max_epoch (int): 总训练轮数
        """

        steps_per_epoch = results["steps_per_epoch"]
        _, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), dpi=300)

        # 绘制损失曲线
        ax1.set_title("Train and Val Loss")
        ax1.plot(results["train_losses"], label="Train", alpha=0.8)
        ax1.plot(
            (np.arange(max_epoch) + 1) * steps_per_epoch,
            results["val_losses"],
            marker="o",
            label="Val",
            linewidth=2,
        )
        ax1.set_xlabel("Training Steps")
        ax1.set_ylabel("Loss")
        ax1.legend()
        ax1.grid(True, alpha=0.1)

        # 绘制准确率曲线
        ax2.set_title("Training and Val Accuracy")
        ax2.plot(results["train_accuracies"], label="Train", alpha=0.8)
        ax2.plot(
            (np.arange(max_epoch) + 1) * steps_per_epoch,
            results["val_accuracies"],
            marker="o",
            label="Val",
            linewidth=2,
        )
        ax2.set_xlabel("Training Steps")
        ax2.set_ylabel("Accuracy")
        ax2.legend()
        ax2.grid(True, alpha=0.1)

        plt.tight_layout()
        plt.show()

    def get_parameters(self) -> Dict[str, np.ndarray]:
        """
        获取模型参数

        Returns:
            Dict[str, np.ndarray]: 包含权重和偏置的字典
        """

        return {"W": self.W.copy(), "b": self.b.copy()}

    def set_parameters(self, W: np.ndarray, b: np.ndarray) -> None:
        """
        设置模型参数

        Args:
            W (np.ndarray): 权重矩阵
            b (np.ndarray): 偏置向量
        """

        assert W.shape == self.W.shape, (
            f"权重形状不匹配: 期望{self.W.shape}, 得到{W.shape}"
        )
        assert b.shape == self.b.shape, (
            f"偏置形状不匹配: 期望{self.b.shape}, 得到{b.shape}"
        )
        self.W = W.copy()
        self.b = b.copy()

## 2、模型实验


### 1、比较使用 momentum 和不使用 momentum 的结果


#### 不使用动量(动量为零)


In [ ]:
model = SoftmaxTrainer()
results_no_momentum = model.train(
    train_data,
    train_labels,
    val_data,
    val_labels,
    batch_size=100,
    max_epoch=10,
    lr=0.001,
    momentum=0.0,
)

In [ ]:
model.plot_curves(results_no_momentum)

In [ ]:
# 测试集评估
_, test_acc = model.evaluate(test_data, test_labels)
print(f"测试集准确率: {test_acc:.2%}")

#### 使用动量


In [ ]:
model = SoftmaxTrainer()
results_with_momentum = model.train(
    train_data,
    train_labels,
    val_data,
    val_labels,
    batch_size=100,
    max_epoch=10,
    lr=0.001,
    momentum=0.9,
)

In [ ]:
model.plot_curves(results_with_momentum)

In [ ]:
# 测试集评估
_, test_acc = model.evaluate(test_data, test_labels)
print(f"测试集准确率: {test_acc:.2%}")

**Comparison of Models Trained With and Without Momentum​：**

- **Training Time**:​​ The total time required to train the model is nearly identical, whether momentum is used or not.

- **Convergence Speed**:​​ The model utilizing momentum demonstrates faster convergence, meaning it reaches a state of minimal loss more quickly during training compared to the model without momentum.

- **​Validation Accuracy** (After 10 Epochs):​​ Upon completing 10 training epochs, the final accuracy achieved by the model with momentum on the validation set is higher than that of the model trained without momentum.

- **Test Accuracy** (After 10 Epochs):​​ Similarly, after 10 epochs, the final accuracy of the model with momentum on the test set is also higher than the model trained without momentum.

- **​Overfitting** **Indication**:​ For both models (with and without momentum), the observed accuracy on the validation set is lower than the accuracy on the test set. This pattern typically indicates the presence of overfitting, meaning the models may have learned patterns specific to the training data that do not generalize perfectly to unseen data (validation set), though they still perform well on the test set.


### 2、比较不同 momentum 结果


In [ ]:
model_1 = SoftmaxTrainer(verbose=False)
for v in [i / 10 for i in range(11)]:
    time_start = time.time()
    results = model_1.train(
        train_data,
        train_labels,
        val_data,
        val_labels,
        batch_size=100,
        max_epoch=10,
        lr=0.01,
        momentum=v,
    )
    dT = time.time() - time_start
    val_acc = results["val_accuracies"][-1]
    _, test_acc = model_1.evaluate(test_data, test_labels)
    print(
        f"momentum: {v}, learning rate: 0.01, Val Acc: {val_acc:.1%}, Test Acc: {test_acc:.1%}, Time: {dT:.2f}s, batch size: 100"
    )

**Comparison of Models with Different Momentum Values:​​**

- **​Training Time:** The training times for models with momentum values ranging from 0.0 to 1.0 show no significant differences.

- **​Validation Accuracy:** As momentum increases from 0.0 to 1.0, validation accuracy initially rises from 92.0% to 93.7%, then declines to 89.7%. The highest validation accuracy (93.7%) is achieved at a momentum of 0.9. Accuracy consistently improves as momentum increases from 0.0 to 0.9 but drops sharply when momentum reaches 1.0.

- **​Test Accuracy:** Similarly, test accuracy increases from 90.2% to 92.2% as momentum goes from 0.0 to 0.9, then falls to 87.5% at a momentum of 1.0. The peak test accuracy (92.2%) also occurs at a momentum of 0.9, followed by a rapid decrease at momentum 1.0.

- **​Conclusion:** Based on these results, a momentum value of 0.9 appears optimal for this experiment.


### 3、比较不同 learning rate 的结果


In [ ]:
model_2 = SoftmaxTrainer(verbose=False)
for lr in [0.0001, 0.001, 0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    time_start = time.time()
    results = model_2.train(
        train_data,
        train_labels,
        val_data,
        val_labels,
        batch_size=100,
        max_epoch=10,
        lr=lr,
        momentum=0.9,
    )
    dT = time.time() - time_start
    val_acc = results["val_accuracies"][-1]
    _, test_acc = model_2.evaluate(test_data, test_labels)
    print(
        f"momentum: 0.9, learning rate: {lr}, Val Acc: {val_acc:.1%}, Test Acc: {test_acc:.1%}, Time: {dT:.2f}s, batch size: 100"
    )

**Comparison of Models with Different Learning Rates:**

- **Training Time**: The training time across models with different learning rates is nearly identical. This suggests that the computational overhead per epoch remains consistent regardless of the learning rate setting, implying that the choice of learning rate primarily influences convergence behavior and accuracy rather than training duration.

- **Convergence Issues with Low Learning Rates**: The model trained with a learning rate of 0.0001 exhibits ​poor convergence, resulting in the lowest accuracy on both the validation and test sets. This aligns with the known phenomenon that excessively low learning rates can lead to extremely slow convergence, causing the model to stagnate in suboptimal solutions or struggle to escape flat regions in the loss landscape.

- **Validation Accuracy**: As the learning rate increases from 0.0001 to 0.01, the model's validation accuracy improves from 88.3% to 93.7%, peaking at a learning rate of 0.01. Further increasing the learning rate beyond 0.01 (up to 1.0) leads to a fluctuating decline in validation accuracy, eventually dropping to 88.7%. This pattern indicates that while moderate learning rates facilitate effective learning, excessively high rates introduce instability and hinder generalization.

- **Test Accuracy**:Similarly, test accuracy rises from 85.4% to 92.2% as the learning rate increases from 0.0001 to 0.01, with the highest accuracy achieved at a learning rate of 0.01. Beyond this point, test accuracy declines fluctuatively, dropping to 86.5% at a learning rate of 1.0. This consistency between validation and test performance underscores the robustness of the optimal learning rate choice.

- **Conclusion**: The experimental results demonstrate that ​a learning rate of 0.01 yields the best performance​ in terms of both validation and test accuracy. Lower learning rates (e.g., 0.0001) suffer from inadequate convergence, while higher rates (e.g., 0.1 to 1.0) cause instability and degradation in model accuracy. Thus, selecting an appropriate learning rate is critical for balancing convergence speed and final model performance.


### 4、比较不同 bach size 的结果


In [ ]:
model_3 = SoftmaxTrainer(verbose=False)
for batch_size in [1, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
    time_start = time.time()
    results = model_3.train(
        train_data,
        train_labels,
        val_data,
        val_labels,
        batch_size=batch_size,
        max_epoch=10,
        lr=0.01,
        momentum=0.9,
    )
    dT = time.time() - time_start
    val_acc = results["val_accuracies"][-1]
    _, test_acc = model_3.evaluate(test_data, test_labels)
    print(
        f"momentum: 0.9, learning rate: 0.01, Val Acc: {val_acc:.1%}, Test Acc: {test_acc:.1%}, Time: {dT:.2f}s, batch size: {batch_size}"
    )

**Comparison of Models with Different Batch Sizes:​​**

- **​Training Time**:​​ The model with a batch size of 1 required the longest training time at 31.53 seconds. Training time decreased significantly to 2.37 seconds with a batch size of 32. As the batch size increased from 32 to 256, training times continued to decrease. For batch sizes ranging from 256 to 4096, the training time showed no significant further reduction and exhibited some fluctuation.

- **​Validation Accuracy**:​​ As the batch size increased from 1 to 64, the validation accuracy rose from 91.2% to a peak of 93.8%. However, as the batch size continued to increase from 64 to 4096, the validation accuracy consistently declined, dropping to 90.1%.

- **​Test Accuracy**:​​ A similar trend was observed for test accuracy. It increased from 89.0% to a maximum of 92.3% as the batch size grew from 1 to 64. Further increasing the batch size from 64 to 4096 resulted in a consistent decrease in test accuracy, which fell to 87.8%.

- **​Conclusion**:​​ Both excessively small (e.g., 1) and excessively large (e.g., >64 up to 4096) batch sizes can be detrimental. An appropriately chosen batch size (like 64 in this case) can significantly reduce training time and improve model performance (achieving the highest validation and test accuracy).
